In [ ]:
from pathlib import Path
import json
import random
import sys

import pandas as pd
from transformer_lens import HookedTransformer

SEED = 12345
random.seed(SEED)

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
model = HookedTransformer.from_pretrained(
    "gpt2-small",
    device="cpu",
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False,
)
tokenizer = model.tokenizer

In [ ]:
NAMES = [
    "John", "Mary", "Tom", "Sarah", "James", "Emily",
    "Robert", "Linda", "Michael", "Jessica", "David", "Susan",
    "William", "Karen", "Richard", "Nancy", "Joseph", "Lisa",
    "Thomas", "Betty", "Charles", "Sandra", "Daniel", "Ashley",
]

OBJECTS = [
    "a bottle of milk",
    "a book",
    "a laptop",
    "a football",
    "a cup of tea",
    "a bag of apples",
    "a bouquet of flowers",
    "a box of chocolates",
    "a set of keys",
    "a sandwich",
]

In [ ]:
def is_single_gpt2_answer_token(name: str) -> bool:
    toks = tokenizer.encode(" " + name, add_special_tokens=False)
    return len(toks) == 1

valid_names = [n for n in NAMES if is_single_gpt2_answer_token(n)]
invalid_names = [n for n in NAMES if n not in valid_names]

print("valid:", valid_names)
print("invalid:", invalid_names)

assert len(valid_names) >= 16, "Need enough single-token GPT-2 names."

In [ ]:
from src.data_ioi import make_ioi_example, validate_answer_tokens

In [ ]:
examples = []
for name_a in valid_names:
    for name_b in valid_names:
        if name_a == name_b:
            continue
        for obj in OBJECTS:
            examples.append(make_ioi_example(name_a, name_b, obj))

random.shuffle(examples)
len(examples), examples[0]

In [ ]:
def token_len(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

bad = []
for ex in examples:
    clean_len = token_len(ex["clean_prompt"])
    corrupt_len = token_len(ex["corrupt_prompt"])
    if clean_len != corrupt_len:
        bad.append((ex["id"], clean_len, corrupt_len))

assert not bad, f"Found clean/corrupt token-length mismatches: {bad[:5]}"

In [ ]:
examples = validate_answer_tokens(model, examples)

In [ ]:
pairs = sorted({(ex["metadata"]["giver_clean"], ex["metadata"]["io_clean"]) for ex in examples})
random.shuffle(pairs)

n = len(pairs)
train_pairs = set(pairs[: int(0.70 * n)])
val_pairs = set(pairs[int(0.70 * n): int(0.85 * n)])
test_pairs = set(pairs[int(0.85 * n):])


def split_for(ex):
    pair = (ex["metadata"]["giver_clean"], ex["metadata"]["io_clean"])
    if pair in train_pairs:
        return "train"
    if pair in val_pairs:
        return "val"
    return "test"


for ex in examples:
    ex["split"] = split_for(ex)

In [ ]:
def write_jsonl(path, rows):
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

write_jsonl(RAW_DIR / "ioi_examples.jsonl", examples)
write_jsonl(PROC_DIR / "ioi_train.jsonl", [e for e in examples if e["split"] == "train"])
write_jsonl(PROC_DIR / "ioi_val.jsonl", [e for e in examples if e["split"] == "val"])
write_jsonl(PROC_DIR / "ioi_test.jsonl", [e for e in examples if e["split"] == "test"])

pd.DataFrame(examples).to_parquet(PROC_DIR / "ioi_examples.parquet", index=False)

In [ ]:
df = pd.DataFrame(examples)

display(df.groupby("split").size())
display(df[[
    "id",
    "split",
    "clean_prompt",
    "corrupt_prompt",
    "clean_correct",
    "corrupt_correct",
    "repair_answer_a",
    "repair_answer_b",
]].head())